In [ ]:
from sklearn.datasets import make_classification

# Generate a synthetic binary classification dataset with two features
X, y = make_classification(n_samples=1000, n_features=2, n_informative=2, n_redundant=0, n_classes=2, random_state=42)

print(f"Shape of features (X): {X.shape}")
print(f"Shape of target (y): {y.shape}")
print(f"Number of classes: {len(set(y))}")
print("First 5 samples of X:\n", X[:5])
print("First 5 samples of y:\n", y[:5])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Sigmoid function for logistic regression
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Assuming X and y are already defined from the previous cell
# For simplicity, let's use the first feature of X for a 2D line (1 feature + intercept)
x_feature = X[:, 0]

# Define a range for the model parameters (slope 'm' and intercept 'b')
m_values = np.linspace(-5, 5, 100)
b_values = np.linspace(-5, 5, 100)
M, B = np.meshgrid(m_values, b_values)

# Calculate MSE for each combination of m and b
Z = np.array([np.mean((sigmoid(m_val * x_feature + b_val) - y)**2) for m_val, b_val in zip(np.ravel(M), np.ravel(B))])
Z = Z.reshape(M.shape)

# Create the 3D plot
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

ax.plot_surface(M, B, Z, cmap='viridis', alpha=0.8)

ax.set_xlabel('Slope (m)')
ax.set_ylabel('Intercept (b)')
ax.set_zlabel('Mean Squared Error (MSE)')
ax.set_title('3D Error Surface (MSE)')

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Sigmoid function for logistic regression
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Binary Cross-Entropy Loss function
def binary_cross_entropy(y_true, y_pred_proba):
    # Clip probabilities to avoid log(0) errors
    y_pred_proba = np.clip(y_pred_proba, 1e-10, 1 - 1e-10)
    return -np.mean(y_true * np.log(y_pred_proba) + (1 - y_true) * np.log(1 - y_pred_proba))

# Assuming X and y are already defined from the previous cell
x_feature = X[:, 0] # Using the first feature for simplicity

# Define a range for the model parameters (slope 'm' and intercept 'b')
m_values = np.linspace(-20, 20, 100)
b_values = np.linspace(-20, 20, 100)
M, B = np.meshgrid(m_values, b_values)

# Calculate BCE for each combination of m and b
Z_bce = np.zeros_like(M, dtype=float)
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        m_val = M[i, j]
        b_val = B[i, j]

        # Predict probabilities using the linear model + sigmoid
        linear_prediction = m_val * x_feature + b_val
        y_pred_proba = sigmoid(linear_prediction)

        # Calculate BCE loss
        Z_bce[i, j] = binary_cross_entropy(y, y_pred_proba)

# Create the 3D plot
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

ax.plot_surface(M, B, Z_bce, cmap='plasma', alpha=0.8)

ax.set_xlabel('Slope (m)')
ax.set_ylabel('Intercept (b)')
ax.set_zlabel('Binary Cross-Entropy Loss')
ax.set_title('3D Error Surface (Binary Cross-Entropy)')

plt.show()

### Applying Gradient Descent for Logistic Regression

Now, let's use gradient descent to find the optimal slope (m) and intercept (b) that minimize the Binary Cross-Entropy loss for our binary classification data. We'll track the loss over iterations to observe convergence and then visualize the final decision boundary.

In [ ]:
# Implement Gradient Descent (Modified to store m and b history)
def gradient_descent(X_feature, y_true, learning_rate=0.1, n_iterations=1000):
    # Initialize parameters randomly
    m = np.random.randn()
    b = np.random.randn()

    loss_history = []
    m_history = []
    b_history = []

    for i in range(n_iterations):
        # Linear prediction
        linear_prediction = m * X_feature + b

        # Predicted probabilities using sigmoid
        y_pred_proba = sigmoid(linear_prediction)

        # Calculate gradients for BCE
        d_m = np.mean((y_pred_proba - y_true) * X_feature)
        d_b = np.mean(y_pred_proba - y_true)

        # Update parameters
        m = m - learning_rate * d_m
        b = b - learning_rate * d_b

        # Calculate and store current loss
        current_loss = binary_cross_entropy(y_true, y_pred_proba)
        loss_history.append(current_loss)
        m_history.append(m)
        b_history.append(b)

        if i % 100 == 0:
            print(f"Iteration {i}: Loss = {current_loss:.4f}")

    return m, b, loss_history, m_history, b_history

# Run Gradient Descent with updated return values
optimal_m, optimal_b, loss_history, m_history, b_history = gradient_descent(x_feature, y, learning_rate=0.5, n_iterations=1000)

print(f"\nOptimal Slope (m): {optimal_m:.4f}")
print(f"Optimal Intercept (b): {optimal_b:.4f}")

In [ ]:
# Plot Loss History
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(loss_history, color='blue')
ax.set_title('Binary Cross-Entropy Loss over Iterations')
ax.set_xlabel('Iteration')
ax.set_ylabel('Loss')
ax.grid(True)
plt.show()

In [ ]:
# Visualize the decision boundary
fig, ax = plt.subplots(figsize=(10, 8))

# Plot the data points (using the first feature of X and y)
# Color points by their class
scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolors='k', alpha=0.7)

# Create decision boundary based on optimal m and b
# For simplicity, we are still using only x_feature (X[:,0]) for the model
# The decision boundary is where sigmoid(m*x + b) = 0.5, which means m*x + b = 0
# So, x_decision = -b / m

# NOTE: This decision boundary is for a single feature model (X[:,0])
# applied to a 2D plot. If the model truly used both features,
# the boundary would be a line defined by w1*x1 + w2*x2 + b = 0.
# Since we only used X[:,0] in gradient descent, we'll plot a vertical line.

# To make it more meaningful for the 2D plot, let's consider a decision boundary
# where the first feature determines the separation given the optimal parameters.
# The linear equation is m*x_1 + b = 0, so x_1 = -b/m

# A more proper 2D decision boundary would involve weights for both features
# and an intercept. Given our current simple GD on one feature, we approximate.

# Let's use a decision threshold on the first feature based on the optimal m and b
# (This is simplified as our GD only used x_feature = X[:,0])
# The actual boundary for P(y=1|X) = 0.5 is where m*X[:,0] + b = 0. So X[:,0] = -b/m

if optimal_m != 0:
    # Calculate the x-value where the decision boundary would be if only x_feature is used
    decision_boundary_x = -optimal_b / optimal_m
    ax.axvline(x=decision_boundary_x, color='green', linestyle='--', label=f'Decision Boundary (X0 = {decision_boundary_x:.2f})')

ax.set_title('Data Points and Simplified Decision Boundary')
ax.set_xlabel('Feature 1 (X[:, 0])')
ax.set_ylabel('Feature 2 (X[:, 1])')
ax.legend()
ax.grid(True)
plt.colorbar(scatter, label='Class (0 or 1)')
plt.show()

### Visualization of Gradient Descent Path on 3D Error Surface

This plot shows the 3D Binary Cross-Entropy error surface, and on top of it, the path taken by the gradient descent algorithm as it optimizes the model parameters (slope `m` and intercept `b`). Each point on the path represents the `(m, b)` values at a specific iteration and their corresponding loss.

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

# Reuse M, B, Z_bce from the previous BCE surface plot cell
# M and B are already defined as meshgrid from m_values and b_values
# Z_bce contains the BCE loss values for the grid

fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# Plot the 3D error surface (BCE)
ax.plot_surface(M, B, Z_bce, cmap='plasma', alpha=0.6, label='BCE Error Surface')

# Plot the gradient descent path
# We need to ensure loss_history, m_history, b_history are available from the previous GD execution
# If they are not, re-run the gradient descent cell (7f1923f3)

# For plotting the path, we need to convert lists to numpy arrays for easier indexing
path_m = np.array(m_history)
path_b = np.array(b_history)
path_loss = np.array(loss_history)

# Plot the trajectory as points
ax.scatter(path_m, path_b, path_loss, color='cyan', s=20, label='GD Path Points')

# Plot the trajectory as a line
ax.plot(path_m, path_b, path_loss, color='blue', linewidth=2, label='GD Path Line')

# Mark the start and end points
ax.scatter(path_m[0], path_b[0], path_loss[0], color='green', s=100, marker='o', label='Start')
ax.scatter(path_m[-1], path_b[-1], path_loss[-1], color='red', s=100, marker='X', label='End (Optimal)')

ax.set_xlabel('Slope (m)')
ax.set_ylabel('Intercept (b)')
ax.set_zlabel('Binary Cross-Entropy Loss')
ax.set_title('Gradient Descent Path on 3D BCE Error Surface')

# Add a legend, but 3D plots don't automatically support line labels well
# You might need to manually create a proxy for the legend if needed for line/surface
# For scatter, labels work directly.
# ax.legend() # This might not work perfectly with plot_surface label

plt.show()